# Testing Model Performance
### Subliminal trials before entire epoch is run

In [1]:
from __future__ import annotations

import re
import json
import inspect
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


In [2]:
# ----------------------------
# Run Controls (cell is intentionally config-only)
# ----------------------------
BIRDS = ["DavidBowie", "Endive"]
SNAPSHOTS_TARGET = [100, 125, 150]
NUMFRAMES_TARGET = [200, 400, 800, 1400]

RUN_CROSS_TRIAL = True
RUN_WITHIN_TRIAL = True
SKIP_MISSING_SNAPSHOTS = True

RUN_BUILD_CLIPS = True
RUN_INFERENCE = True
RUN_EVALUATION = True

ALLOW_HIGH_LEVEL_FALLBACK = True
OVERWRITE_CLIPS = False
OVERWRITE_INFERENCE = False

VIDEO_CODEC = "MJPG"
VIDEO_FPS = 500.0

THRESHOLD_MODE = "rmse"  # rmse|fixed
FIXED_THRESHOLD_PX = 8.0

NOTEBOOK_DIR = Path.cwd()
TESTING_ROOT = NOTEBOOK_DIR.parent
DEEPLABCUT_ROOT = TESTING_ROOT / "DeepLabCut"

print(f"Testing root: {TESTING_ROOT}")
print(f"DeepLabCut root: {DEEPLABCUT_ROOT}")

Testing root: c:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing
DeepLabCut root: c:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut


In [22]:
# ----------------------------
# Pair Mode Expansion
# ----------------------------
TRIAL_MAP = {
    "DavidBowie": [15, 17],
    "Endive": [32],
}


def build_trial_pairs(bird: str) -> list[tuple[int, int]]:
    trials = TRIAL_MAP.get(bird, [])
    pairs: list[tuple[int, int]] = []

    if RUN_CROSS_TRIAL and len(trials) >= 2:
        for t in trials:
            for e in trials:
                if t != e:
                    pairs.append((t, e))

    if RUN_WITHIN_TRIAL:
        for t in trials:
            if t == 17:
                continue
            else:
                pairs.append((t, t))

    # deterministic order + unique
    pairs = sorted(set(pairs), key=lambda x: (x[0], x[1]))
    return pairs


all_pair_rows: list[dict[str, Any]] = []
for bird in BIRDS:
    for train_trial, eval_trial in build_trial_pairs(bird):
        if train_trial == 17 and eval_trial == 17:
            print(f"Skipping pair {bird} T{train_trial}-T{eval_trial} due to missing snapshots")
            continue
        else:
            all_pair_rows.append(
                {
                    "bird": bird,
                    "train_trial": train_trial,
                    "eval_trial": eval_trial,
                    "pair_slug": f"TrainT{train_trial}_EvalT{eval_trial}",
                }
            )
    

pair_df = pd.DataFrame(all_pair_rows)
pair_df = pair_df[~((pair_df['train_trial'] == 17) & (pair_df['eval_trial'] == 17))]  # exclude T17-T17 due to missing snapshots
display(pair_df)

,bird,train_trial,eval_trial,pair_slug
0,DavidBowie,15,15,TrainT15_EvalT15
1,DavidBowie,15,17,TrainT15_EvalT17
2,DavidBowie,17,15,TrainT17_EvalT15
3,Endive,32,32,TrainT32_EvalT32


In [23]:
# ----------------------------
# Shared Helpers: Naming + Discovery
# ----------------------------


def pair_slug(train_trial: int, eval_trial: int) -> str:
    return f"TrainT{train_trial}_EvalT{eval_trial}"


def bird_root(bird: str) -> Path:
    return DEEPLABCUT_ROOT / bird


def batch_root(bird: str) -> Path:
    return bird_root(bird) / "ExperimentEval" / "UFBatch"


def model_build_root(bird: str) -> Path:
    return bird_root(bird) / "ModelBuildExperiments"


def discover_model_configs_for_trial(bird: str, train_trial: int) -> list[Path]:
    root = model_build_root(bird)
    if not root.exists():
        return []

    configs = sorted(root.glob(f"{bird}_n*_T{train_trial}/Canari-Tyler-*/config.yaml"))
    return [c for c in configs if c.exists()]


def extract_numframes_from_model_dir(model_dir_name: str) -> int | None:
    m = re.search(r"_n(\d+)_T\d+$", model_dir_name)
    return int(m.group(1)) if m else None

In [73]:
# ----------------------------
# Shared Helpers: Eval Assets + Clips
# ----------------------------


def resolve_eval_train_dir(bird: str, eval_trial: int) -> Path:
    d = bird_root(bird) / f"TrainingData_T{eval_trial}"
    if d.exists():
        return d
    raise FileNotFoundError(f"Could not find eval training dir for {bird} T{eval_trial}: {d}")


def resolve_cam_dir(eval_train_dir: Path, eval_trial: int, cam_idx: int) -> Path:
    expected = eval_train_dir / f"Cam{cam_idx}_Img00UND_Trial{eval_trial}"
    if expected.exists():
        return expected
    matches = sorted(eval_train_dir.rglob(f"Cam{cam_idx}_Img00UND_Trial{eval_trial}"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Missing Cam{cam_idx} eval image dir under {eval_train_dir}")


def resolve_truth_csv(eval_train_dir: Path, eval_trial: int) -> Path:
    candidates = sorted(eval_train_dir.rglob(f"*Trial{eval_trial}*.csv"))
    if not candidates:
        candidates = sorted(eval_train_dir.rglob("*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No truth CSV under {eval_train_dir}")
    return candidates[0]


def parse_frame_number_from_stem(stem: str) -> int | None:
    m = re.search(r"(\d+)(?!.*\d)", stem)
    return int(m.group(1)) if m else None


def collect_images(image_dir: Path) -> list[Path]:
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    images = [p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in exts]
    return sorted(images, key=lambda p: (parse_frame_number_from_stem(p.stem) is None, parse_frame_number_from_stem(p.stem) or 0, p.name))

In [18]:
# ----------------------------
# Shared Helpers: Video Build
# ----------------------------


def build_video_from_images(
    image_paths: list[Path],
    out_video: Path,
    fps: float = VIDEO_FPS,
    codec: str = VIDEO_CODEC,
    overwrite: bool = False,
) -> dict[str, Any]:
    try:
        import cv2
    except Exception as exc:
        raise ImportError("OpenCV (cv2) is required to build evaluation clips.") from exc

    if not image_paths:
        raise ValueError(f"No image paths provided for {out_video}")

    if out_video.exists() and not overwrite:
        frame_lookup = [parse_frame_number_from_stem(p.stem) or (i + 1) for i, p in enumerate(image_paths)]
        return {"video": out_video, "status": "reused", "n_frames": len(frame_lookup), "frame_lookup": frame_lookup}

    first = cv2.imread(str(image_paths[0]))
    if first is None:
        raise FileNotFoundError(f"Could not read first image: {image_paths[0]}")

    h, w = first.shape[:2]
    out_video.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(out_video), cv2.VideoWriter_fourcc(*codec), float(fps), (w, h))

    n_written = 0
    frame_lookup: list[int] = []
    for img_path in image_paths:
        frame = cv2.imread(str(img_path))
        if frame is None:
            continue
        if frame.shape[:2] != (h, w):
            frame = cv2.resize(frame, (w, h))
        writer.write(frame)
        n_written += 1
        frame_lookup.append(parse_frame_number_from_stem(img_path.stem) or n_written)

    writer.release()
    return {"video": out_video, "status": "created", "n_frames": n_written, "frame_lookup": frame_lookup}

In [74]:
# ----------------------------
# Discovery Execution: Bird/Pair/Model Inventory
# ----------------------------

inventory_rows: list[dict[str, Any]] = []

for bird in BIRDS:
    for train_trial, eval_trial in build_trial_pairs(bird):
        cfgs = discover_model_configs_for_trial(bird, train_trial)
        eval_dir = resolve_eval_train_dir(bird, eval_trial)
        truth_csv = resolve_truth_csv(eval_dir, eval_trial)

        for cfg in cfgs:
            model_dir = cfg.parent.parent.name
            numframes = extract_numframes_from_model_dir(model_dir)
            if numframes is None or (NUMFRAMES_TARGET and numframes not in NUMFRAMES_TARGET):
                continue

            inventory_rows.append(
                {
                    "bird": bird,
                    "train_trial": train_trial,
                    "eval_trial": eval_trial,
                    "pair_slug": pair_slug(train_trial, eval_trial),
                    "model_dir": model_dir,
                    "numframes": numframes,
                    "config_path": str(cfg),
                    "eval_dir": str(eval_dir),
                    "truth_csv": str(truth_csv),
                }
            )

inventory_df = pd.DataFrame(inventory_rows).sort_values(["bird", "train_trial", "eval_trial", "numframes"]).reset_index(drop=True)
print(f"Discovered inventory rows: {len(inventory_df)}")
display(inventory_df.head(50))

Discovered inventory rows: 14


,bird,train_trial,eval_trial,pair_slug,model_dir,numframes,config_path,eval_dir,truth_csv
0,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...
1,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...
2,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...
3,DavidBowie,15,17,TrainT15_EvalT17,DavidBowie_n200_T15,200,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...
4,DavidBowie,15,17,TrainT15_EvalT17,DavidBowie_n400_T15,400,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...
5,DavidBowie,15,17,TrainT15_EvalT17,DavidBowie_n800_T15,800,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...
6,DavidBowie,17,15,TrainT17_EvalT15,DavidBowie_n200_T17,200,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...
7,DavidBowie,17,15,TrainT17_EvalT15,DavidBowie_n400_T17,400,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...
8,DavidBowie,17,15,TrainT17_EvalT15,DavidBowie_n800_T17,800,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...
9,DavidBowie,17,15,TrainT17_EvalT15,DavidBowie_n1400_T17,1400,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...


In [76]:
for i in range(13):
    print(inventory_df['truth_csv'][i])

c:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TrainingData_T15\CollectedData_Trial15_DB.csv
c:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TrainingData_T15\CollectedData_Trial15_DB.csv
c:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TrainingData_T15\CollectedData_Trial15_DB.csv
c:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TrainingData_T17\2Dpoints_2025_08_25_DavidBowie_Subset_Trial17_UND_matches.csv
c:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TrainingData_T17\2Dpoints_2025_08_25_DavidBowie_Subset_Trial17_UND_matches.csv
c:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TrainingData_T17\2Dpoints_2025_08_25_DavidBowie_Subset_Trial17_UND_matches.csv
c:\Users\Salle-Cineradio\Documents\MachineLearnin

In [25]:
# ----------------------------
# Snapshot Helpers
# ----------------------------


def discover_snapshot_files(config_path: Path) -> list[Path]:
    model_root = config_path.parent
    candidates = sorted(model_root.rglob("snapshot-*.pt"))
    if candidates:
        return candidates
    # fallback pattern used in older layouts
    return sorted(model_root.rglob("*snapshot*.pt"))


def parse_snapshot_number(snapshot_path: Path) -> int | None:
    m = re.search(r"snapshot[-_](\d+)", snapshot_path.stem)
    return int(m.group(1)) if m else None


def select_snapshot_files(snapshot_files: list[Path], targets: list[int]) -> tuple[dict[int, Path], list[int]]:
    by_num: dict[int, Path] = {}
    for s in snapshot_files:
        n = parse_snapshot_number(s)
        if n is not None and n not in by_num:
            by_num[n] = s

    missing = [n for n in targets if n not in by_num]
    selected = {n: by_num[n] for n in targets if n in by_num}
    return selected, missing

In [27]:
# ----------------------------
# Snapshot Availability Table
# ----------------------------

snapshot_rows: list[dict[str, Any]] = []

for _, row in inventory_df.iterrows():
    cfg = Path(row["config_path"])
    files = discover_snapshot_files(cfg) 
    selected, missing = select_snapshot_files(files, SNAPSHOTS_TARGET)

    for n in SNAPSHOTS_TARGET:
        snapshot_rows.append(
            {
                "bird": row["bird"],
                "train_trial": int(row["train_trial"]),
                "eval_trial": int(row["eval_trial"]),
                "pair_slug": row["pair_slug"],
                "model_dir": row["model_dir"],
                "numframes": int(row["numframes"]),
                "snapshot_num": n,
                "found": n in selected,
                "snapshot_path": str(selected[n]) if n in selected else None,
                "status": "missing" if n in missing else "ready",
            }
        )

snapshot_df = pd.DataFrame(snapshot_rows).sort_values(
    ["bird", "train_trial", "eval_trial", "numframes", "snapshot_num"]
).reset_index(drop=True)

print("Snapshot availability:")
display(snapshot_df.head(100))
print(snapshot_df["status"].value_counts(dropna=False))

Snapshot availability:


,bird,train_trial,eval_trial,pair_slug,model_dir,numframes,snapshot_num,found,snapshot_path,status
0,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,100,True,c:\Users\Salle-Cineradio\Documents\MachineLear...,ready
1,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,125,True,c:\Users\Salle-Cineradio\Documents\MachineLear...,ready
2,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,150,True,c:\Users\Salle-Cineradio\Documents\MachineLear...,ready
3,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,100,True,c:\Users\Salle-Cineradio\Documents\MachineLear...,ready
4,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,125,True,c:\Users\Salle-Cineradio\Documents\MachineLear...,ready
5,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,150,True,c:\Users\Salle-Cineradio\Documents\MachineLear...,ready
6,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,100,True,c:\Users\Salle-Cineradio\Documents\MachineLear...,ready
7,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,125,True,c:\Users\Salle-Cineradio\Documents\MachineLear...,ready
8,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,150,True,c:\Users\Salle-Cineradio\Documents\MachineLear...,ready
9,DavidBowie,15,17,TrainT15_EvalT17,DavidBowie_n200_T15,200,100,True,c:\Users\Salle-Cineradio\Documents\MachineLear...,ready


status
ready    42
Name: count, dtype: int64


In [28]:
# ----------------------------
# Backend Probe: Low-level PyTorch APIs
# ----------------------------

backend_info: dict[str, Any] = {
    "has_deeplabcut": False,
    "has_pytorch_api_module": False,
    "low_level_supported": False,
    "analyze_videos_signature": None,
}

try:
    import deeplabcut

    backend_info["has_deeplabcut"] = True

    try:
        from deeplabcut.pose_estimation_pytorch import apis as dlc_pt_apis

        backend_info["has_pytorch_api_module"] = True
        if hasattr(dlc_pt_apis, "analyze_videos"):
            backend_info["low_level_supported"] = True
            backend_info["analyze_videos_signature"] = str(inspect.signature(dlc_pt_apis.analyze_videos))
    except Exception as exc:
        backend_info["pytorch_api_error"] = str(exc)
except Exception as exc:
    backend_info["deeplabcut_import_error"] = str(exc)

print(json.dumps(backend_info, indent=2))

Loading DLC 3.0.0rc13...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)
{
  "has_deeplabcut": true,
  "has_pytorch_api_module": true,
  "low_level_supported": true,
  "analyze_videos_signature": "(config: 'str', videos: 'str | list[str]', videotype: 'str | None' = None, shuffle: 'int' = 1, trainingsetindex: 'int' = 0, save_as_csv: 'bool' = False, in_random_order: 'bool' = False, snapshot_index: 'int | str | None' = None, detector_snapshot_index: 'int | str | None' = None, device: 'str | None' = None, destfolder: 'str | None' = None, batch_size: 'int | None' = None, detector_batch_size: 'int | None' = None, dynamic: 'tuple[bool, float, int]' = (False, 0.5, 10), ctd_conditions: 'dict | CondFromModel | None' = None, ctd_tracking: 'bool | dict | CTDTrackingConfig' = False, top_down_dynamic: 'dict | None' = None, modelprefix: 'str' = '', use_shelve: 'bool' = False, robust_nframes: 'bool' = False, transform: 'A.Compose | None' = None, auto_track: '

In [32]:
# ----------------------------
# Build Run Queue (one row = one snapshot run)
# ----------------------------

run_rows: list[dict[str, Any]] = []

for _, srow in snapshot_df.iterrows():
    if not bool(srow["found"]):
        if SKIP_MISSING_SNAPSHOTS:
            continue
        raise FileNotFoundError(
            f"Missing requested snapshot {srow['snapshot_num']} for {srow['model_dir']} ({srow['pair_slug']})."
        )

    out_dir = (
        batch_root(str(srow["bird"]))
        / str(srow["pair_slug"])
        / f"{srow['bird']}_n{int(srow['numframes'])}_T{int(srow['train_trial'])}_snapshot{int(srow['snapshot_num'])}"
    )

    run_rows.append(
        {
            "bird": str(srow["bird"]),
            "train_trial": int(srow["train_trial"]),
            "eval_trial": int(srow["eval_trial"]),
            "pair_slug": str(srow["pair_slug"]),
            "model_dir": str(srow["model_dir"]),
            "numframes": int(srow["numframes"]),
            "snapshot_num": int(srow["snapshot_num"]),
            "snapshot_path": str(srow["snapshot_path"]),
            "output_dir": str(out_dir),
            "status": "queued",
        }
    )

run_queue_df = pd.DataFrame(run_rows).sort_values(
    ["bird", "train_trial", "eval_trial", "numframes", "snapshot_num"]
).reset_index(drop=True)

print(f"Run queue rows: {len(run_queue_df)}")
display(run_queue_df.head(100))
# we do not wont to run train 17 eval 17 because of the bad quality of the data but we do want to run train 17 eval 15 and train 15 eval 17
run_queue_df = run_queue_df[(run_queue_df["train_trial"] != 17) | (run_queue_df["eval_trial"] != 17)]
display(run_queue_df.head(100))
print(run_queue_df['output_dir'][0])

Run queue rows: 42


,bird,train_trial,eval_trial,pair_slug,model_dir,numframes,snapshot_num,snapshot_path,output_dir,status
0,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
1,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
2,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
3,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
4,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
5,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
6,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
7,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
8,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
9,DavidBowie,15,17,TrainT15_EvalT17,DavidBowie_n200_T15,200,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued


,bird,train_trial,eval_trial,pair_slug,model_dir,numframes,snapshot_num,snapshot_path,output_dir,status
0,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
1,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
2,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
3,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
4,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
5,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
6,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
7,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
8,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
9,DavidBowie,15,17,TrainT15_EvalT17,DavidBowie_n200_T15,200,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued


c:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\ExperimentEval\UFBatch\TrainT15_EvalT15\DavidBowie_n200_T15_snapshot100


In [ ]:
# ----------------------------
# Execution Helpers: Clip Prep + Inference Adapter
# ----------------------------


def get_inventory_row(bird: str, train_trial: int, eval_trial: int, numframes: int) -> pd.Series:
    m = inventory_df[
        (inventory_df["bird"] == bird)
        & (inventory_df["train_trial"] == train_trial)
        & (inventory_df["eval_trial"] == eval_trial)
        & (inventory_df["numframes"] == numframes)
    ]
    if m.empty:
        raise KeyError(f"Inventory row missing for {bird}, T{train_trial}->T{eval_trial}, n{numframes}")
    return m.iloc[0]


def ensure_eval_clips_for_run(bird: str, eval_trial: int, pair_slug_text: str) -> tuple[list[Path], dict[str, list[int]]]:
    eval_dir = resolve_eval_train_dir(bird, eval_trial)
    cam_dirs = [resolve_cam_dir(eval_dir, eval_trial, 1), resolve_cam_dir(eval_dir, eval_trial, 2)]

    clip_dir = batch_root(bird) / pair_slug_text / "sv"
    clip_dir.mkdir(parents=True, exist_ok=True)

    subset_videos: list[Path] = []
    frame_lookup_by_cam: dict[str, list[int]] = {}

    for cam_dir in cam_dirs:
        camera = "cam2" if "cam2" in cam_dir.name.lower() else "cam1"
        images = collect_images(cam_dir)
        clip_path = clip_dir / f"db_{camera}_T{eval_trial}.avi"
        info = build_video_from_images(images, clip_path, overwrite=OVERWRITE_CLIPS and RUN_BUILD_CLIPS)
        subset_videos.append(Path(info["video"]))
        frame_lookup_by_cam[camera] = list(info["frame_lookup"])

    return subset_videos, frame_lookup_by_cam


def run_snapshot_inference_one(run_row: pd.Series) -> dict[str, Any]:
    bird = str(run_row["bird"])
    train_trial = int(run_row["train_trial"])
    eval_trial = int(run_row["eval_trial"])
    pair = str(run_row["pair_slug"])
    numframes = int(run_row["numframes"])
    snapshot_num = int(run_row["snapshot_num"])
    snapshot_path = Path(str(run_row["snapshot_path"]))

    inv = get_inventory_row(bird, train_trial, eval_trial, numframes)
    config_path = Path(str(inv["config_path"]))
    output_dir = Path(str(run_row["output_dir"]))
    output_dir.mkdir(parents=True, exist_ok=True)

    subset_videos, _ = ensure_eval_clips_for_run(bird, eval_trial, pair)

    if not RUN_INFERENCE:
        return {**run_row.to_dict(), "status": "skipped_inference_disabled", "error": None}

    # Skip rerun if outputs already exist and overwrite disabled.
    existing_csv = list(output_dir.glob("*DLC*.csv"))
    if existing_csv and not OVERWRITE_INFERENCE:
        return {**run_row.to_dict(), "status": "skipped_exists", "error": None}

    try:
        import deeplabcut

        # Preferred: low-level pytorch API.
        used_backend = "low_level_pytorch"
        low_level_ok = False
        try:
            
            if hasattr(dlc_pt_apis, "analyze_videos"):
                # Best-effort call pattern with snapshot path.
                dlc_pt_apis.analyze_videos(
                    config=str(config_path),
                    videos=[str(v) for v in subset_videos],
                    destfolder=str(output_dir),
                    videotype=".avi",
                    save_as_csv=True,
                    model_file=str(snapshot_path),
                )
                low_level_ok = True
        except Exception as low_exc:
            if not ALLOW_HIGH_LEVEL_FALLBACK:
                raise RuntimeError(f"Low-level inference failed and fallback disabled: {low_exc}") from low_exc
            used_backend = "high_level_fallback"

        if not low_level_ok:
            # Fallback route if low-level call signature differs in this DLC version.
            deeplabcut.analyze_videos(
                config=str(config_path),
                videos=[str(v) for v in subset_videos],
                destfolder=str(output_dir),
                videotype=".avi",
                save_as_csv=True,
            )

        return {
            **run_row.to_dict(),
            "status": "done",
            "backend": used_backend,
            "config_path": str(config_path),
            "n_csv": len(list(output_dir.glob("*DLC*.csv"))),
            "error": None,
        }
    except Exception as exc:
        return {**run_row.to_dict(), "status": "failed", "error": str(exc)}

In [38]:
davidbowie_run_queue_df = run_queue_df[run_queue_df['bird'] == 'DavidBowie'].reset_index(drop=True)
davidbowie_run_queue_df

,bird,train_trial,eval_trial,pair_slug,model_dir,numframes,snapshot_num,snapshot_path,output_dir,status
0,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
1,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
2,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
3,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
4,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
5,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
6,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
7,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
8,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued
9,DavidBowie,15,17,TrainT15_EvalT17,DavidBowie_n200_T15,200,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,queued


In [45]:
from deeplabcut.pose_estimation_pytorch import apis as dlc_pt_apis
# ----------------------------
# Execution: Run Queue David Bowie Only
# ----------------------------

run_results: list[dict[str, Any]] = []

for _, qrow in davidbowie_run_queue_df.iterrows():
    result = run_snapshot_inference_one(qrow)
    run_results.append(result)

run_results_df = pd.DataFrame(run_results)
print("Run results status:")
print(run_results_df["status"].value_counts(dropna=False))
display(run_results_df.head(100))
run_results_df["status"] = 'done'

Run results status:
status
skipped_exists    30
Name: count, dtype: int64


,bird,train_trial,eval_trial,pair_slug,model_dir,numframes,snapshot_num,snapshot_path,output_dir,status,error
0,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,skipped_exists,None
1,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,skipped_exists,None
2,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n200_T15,200,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,skipped_exists,None
3,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,skipped_exists,None
4,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,skipped_exists,None
5,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n400_T15,400,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,skipped_exists,None
6,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,skipped_exists,None
7,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,125,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,skipped_exists,None
8,DavidBowie,15,15,TrainT15_EvalT15,DavidBowie_n800_T15,800,150,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,skipped_exists,None
9,DavidBowie,15,17,TrainT15_EvalT17,DavidBowie_n200_T15,200,100,c:\Users\Salle-Cineradio\Documents\MachineLear...,c:\Users\Salle-Cineradio\Documents\MachineLear...,skipped_exists,None


In [ ]:
endive_run_queue_df = run_queue_df[run_queue_df["bird"] == "Endive"].copy()
endive_run_queue_df = endive_run_queue_df.reset_index(drop=True)
endive_run_queue_df

In [ ]:
# ----------------------------
# Execution: Run Queue: Endive Only
# ----------------------------

run_results: list[dict[str, Any]] = []

for _, qrow in endive_run_queue_df.iterrows():
    result = run_snapshot_inference_one(qrow)
    run_results.append(result)

run_results_df = pd.DataFrame(run_results)
print("Run results status:")
print(run_results_df["status"].value_counts(dropna=False))
display(run_results_df.head(100))

In [39]:
# ----------------------------
# Evaluation Helpers: Truth + Prediction Parsing
# ----------------------------


def infer_camera_from_name(name: str) -> str:
    low = name.lower()
    if "cam2" in low:
        return "cam2"
    return "cam1"


def truth_csv_to_long(truth_csv_path: Path) -> pd.DataFrame:
    truth_df = pd.read_csv(truth_csv_path)
    rows = []
    for col in truth_df.columns:
        m = re.match(r"(?P<bodypart>.+)_cam(?P<cam>[12])_(?P<coord>[XY])$", col)
        if m is None:
            continue
        rows.append((m.group("bodypart"), f"cam{m.group('cam')}", m.group("coord").lower(), col))

    if not rows:
        raise ValueError(f"Could not parse truth columns in {truth_csv_path}")

    meta = pd.DataFrame(rows, columns=["bodypart", "camera", "coord", "column"])
    xmeta = meta[meta["coord"] == "x"].rename(columns={"column": "xcol"})
    ymeta = meta[meta["coord"] == "y"].rename(columns={"column": "ycol"})
    pairs = xmeta.merge(ymeta[["bodypart", "camera", "ycol"]], on=["bodypart", "camera"], how="inner")

    frame_ids = np.arange(1, len(truth_df) + 1)
    parts = []
    for _, r in pairs.iterrows():
        parts.append(
            pd.DataFrame(
                {
                    "frame_id": frame_ids,
                    "bodypart": r["bodypart"],
                    "camera": r["camera"],
                    "x_true": pd.to_numeric(truth_df[r["xcol"]], errors="coerce"),
                    "y_true": pd.to_numeric(truth_df[r["ycol"]], errors="coerce"),
                }
            )
        )
    return pd.concat(parts, ignore_index=True)


def prediction_csv_to_long(pred_csv_path: Path, frame_lookup: list[int]) -> pd.DataFrame:
    wide = pd.read_csv(pred_csv_path, header=[0, 1, 2], index_col=0)
    camera = infer_camera_from_name(pred_csv_path.name)

    bodyparts = pd.Index(wide.columns.get_level_values(1)).unique()
    out = []
    for bp in bodyparts:
        bp_cols = wide.xs(bp, axis=1, level=1, drop_level=False)
        coords = bp_cols.columns.get_level_values(2).astype(str).str.lower().tolist()
        if "x" not in coords or "y" not in coords:
            continue

        xcol = bp_cols.columns[coords.index("x")]
        ycol = bp_cols.columns[coords.index("y")]
        if "likelihood" in coords:
            lcol = bp_cols.columns[coords.index("likelihood")]
            like = pd.to_numeric(bp_cols[lcol], errors="coerce").to_numpy(dtype=float)
        else:
            like = np.full(bp_cols.shape[0], np.nan)

        subset_idx = pd.to_numeric(pd.Index(wide.index), errors="coerce").astype("Int64")
        frame_ids = [frame_lookup[int(i)] if pd.notna(i) and 0 <= int(i) < len(frame_lookup) else np.nan for i in subset_idx]

        out.append(
            pd.DataFrame(
                {
                    "frame_id": frame_ids,
                    "bodypart": str(bp),
                    "camera": camera,
                    "x_pred": pd.to_numeric(bp_cols[xcol], errors="coerce").to_numpy(dtype=float),
                    "y_pred": pd.to_numeric(bp_cols[ycol], errors="coerce").to_numpy(dtype=float),
                    "likelihood": like,
                }
            )
        )

    if not out:
        raise ValueError(f"No usable prediction columns in {pred_csv_path}")
    return pd.concat(out, ignore_index=True)

In [87]:
# ----------------------------
# Evaluation Execution
# ----------------------------

summary_rows: list[dict[str, Any]] = []
error_rows_all: list[pd.DataFrame] = []

if RUN_EVALUATION:
    for _, r in run_results_df.iterrows():
        if str(r.get("status", "")) != "done":
            continue

        try:
            bird = str(r["bird"])
            train_trial = int(r["train_trial"])
            eval_trial = int(r["eval_trial"])
            numframes = int(r["numframes"])
            snapshot_num = int(r["snapshot_num"])
            pair = str(r["pair_slug"])
            out_dir = Path(str(r["output_dir"]))
            print(f"Evaluating {bird} T{train_trial}->T{eval_trial}, n{numframes}, snapshot {snapshot_num}")
            inv = get_inventory_row(bird, train_trial, eval_trial, numframes)
            # print(get_inventory_row(bird, train_trial, eval_trial, numframes))
            # print(str(inv["truth_csv"]))
            truth_csv = Path(str(inv["truth_csv"]))
            truth_long = truth_csv_to_long(truth_csv)
            

            # rebuild frame lookup from eval clip images (cam-wise lookup unified by index assumption)
            eval_dir = resolve_eval_train_dir(bird, eval_trial)
            # print(f"Resolving frame lookup for {bird} T{eval_trial} from {eval_dir}")
            cam1_dir = resolve_cam_dir(eval_dir, eval_trial, 1)
            # print(f"Collecting images from {cam1_dir} for frame lookup")
            frame_lookup = [parse_frame_number_from_stem(p.stem) or (i + 1) for i, p in enumerate(collect_images(cam1_dir))]

            pred_csvs = sorted(out_dir.glob("*DLC*.csv"))
            if not pred_csvs:
                continue

            pred_long = pd.concat([prediction_csv_to_long(p, frame_lookup) for p in pred_csvs], ignore_index=True)
            # pred_long['frame_id'] = pred_long['frame_id'] - 3538 # make integer and align with truth frame ids
            # pred_long["frame_id"] = pred_long["frame_id"].astype("Int64")
            # print(pred_long.head())
            # print(truth_long.head())
            # print(f"Parsed predictions for {bird} T{eval_trial} from {len(pred_csvs)} CSVs, {len(pred_long)} total points")
            if pred_long['frame_id'][0] != 1:
                pred_long['frame_id'] = pred_long['frame_id'] - 3538 # make integer and align with truth frame ids
                pred_long["frame_id"] = pred_long["frame_id"].astype("Int64")

            # print(truth_long['frame_id'][0])
            merged = pred_long.merge(truth_long, on=["frame_id", "bodypart", "camera"], how="inner")
            # print(merged.head())
            merged = merged.dropna(subset=["frame_id", "x_pred", "y_pred", "x_true", "y_true"]).copy()
            
            if merged.empty:
                continue

            merged["distance_px"] = np.sqrt((merged["x_pred"] - merged["x_true"]) ** 2 + (merged["y_pred"] - merged["y_true"]) ** 2)
            # print(merged.head())
            rmse = float(np.sqrt(np.mean(np.square(merged["distance_px"]))))
            threshold_px = rmse if THRESHOLD_MODE == "rmse" else float(FIXED_THRESHOLD_PX)
            merged["within_threshold"] = merged["distance_px"] <= threshold_px
            

            pct_points = 100.0 * float(merged["within_threshold"].mean())
            expected_per_frame = pred_long[["camera", "bodypart"]].drop_duplicates().shape[0]
            frame_stats = merged.groupby("frame_id", as_index=False).agg(
                n_points=("within_threshold", "size"),
                all_within=("within_threshold", "all"),
            )
            # print(frame_stats.head())
            frame_stats["frame_pass"] = frame_stats["all_within"] & (frame_stats["n_points"] == expected_per_frame)
            pct_frames = 100.0 * float(frame_stats["frame_pass"].mean())
            # print(pct_frames)
            mean_likelihood = float(np.nanmean(merged["likelihood"])) if "likelihood" in merged.columns else np.nan

            summary_rows.append(
                {
                    "bird": bird,
                    "train_trial": train_trial,
                    "eval_trial": eval_trial,
                    "pair_slug": pair,
                    "numframes": numframes,
                    "snapshot_num": snapshot_num,
                    "model_dir": str(r["model_dir"]),
                    "output_dir": str(out_dir),
                    "rmse_px": rmse,
                    "threshold_px": threshold_px,
                    "percent_frames_all_points_within_threshold": pct_frames,
                    "percent_points_within_threshold": pct_points,
                    "mean_likelihood": mean_likelihood,
                    "n_points": int(merged.shape[0]),
                }
            )
            # print(summary_rows[-1])

            err = merged[["camera", "bodypart", "frame_id", "distance_px", "likelihood"]].copy()
            err["bird"] = bird
            err["train_trial"] = train_trial
            err["eval_trial"] = eval_trial
            err["pair_slug"] = pair
            err["numframes"] = numframes
            err["snapshot_num"] = snapshot_num
            err["model_dir"] = str(r["model_dir"])
            error_rows_all.append(err)

        except Exception as exc:
            summary_rows.append(
                {
                    "bird": str(r.get("bird")),
                    "train_trial": int(r.get("train_trial")),
                    "eval_trial": int(r.get("eval_trial")),
                    "pair_slug": str(r.get("pair_slug")),
                    "numframes": int(r.get("numframes")),
                    "snapshot_num": int(r.get("snapshot_num")),
                    "model_dir": str(r.get("model_dir")),
                    "output_dir": str(r.get("output_dir")),
                    "status": "eval_failed",
                    "error": str(exc),
                }
            )

summary_df = pd.DataFrame(summary_rows)
errors_df = pd.concat(error_rows_all, ignore_index=True) if error_rows_all else pd.DataFrame()

print(f"Evaluation summary rows: {len(summary_df)}")
display(summary_df.head(100))
print(f"Evaluation error rows: {len(errors_df)}")


Evaluating DavidBowie T15->T15, n200, snapshot 100
Evaluating DavidBowie T15->T15, n200, snapshot 125
Evaluating DavidBowie T15->T15, n200, snapshot 150
Evaluating DavidBowie T15->T15, n400, snapshot 100
Evaluating DavidBowie T15->T15, n400, snapshot 125
Evaluating DavidBowie T15->T15, n400, snapshot 150
Evaluating DavidBowie T15->T15, n800, snapshot 100
Evaluating DavidBowie T15->T15, n800, snapshot 125
Evaluating DavidBowie T15->T15, n800, snapshot 150
Evaluating DavidBowie T15->T17, n200, snapshot 100
Evaluating DavidBowie T15->T17, n200, snapshot 125
Evaluating DavidBowie T15->T17, n200, snapshot 150
Evaluating DavidBowie T15->T17, n400, snapshot 100
Evaluating DavidBowie T15->T17, n400, snapshot 125
Evaluating DavidBowie T15->T17, n400, snapshot 150
Evaluating DavidBowie T15->T17, n800, snapshot 100
Evaluating DavidBowie T15->T17, n800, snapshot 125
Evaluating DavidBowie T15->T17, n800, snapshot 150
Evaluating DavidBowie T17->T15, n200, snapshot 100
Evaluating DavidBowie T17->T15,

MemoryError: Unable to allocate 24.5 MiB for an array with shape (1, 3205476) and data type int64

In [ ]:
# ----------------------------
# Exports
# ----------------------------

exports_root = TESTING_ROOT / "DeepLabCut" / "DB_END_COMBINED_EVAL"
exports_root.mkdir(parents=True, exist_ok=True)

run_manifest_csv = exports_root / "subepochs_run_manifest.csv"
summary_csv = exports_root / "subepochs_summary.csv"
errors_csv = exports_root / "subepochs_errors.csv"

if 'run_results_df' in globals() and not run_results_df.empty:
    run_results_df.to_csv(run_manifest_csv, index=False)
    print(f"Wrote run manifest: {run_manifest_csv}")

if 'summary_df' in globals() and not summary_df.empty:
    summary_df.to_csv(summary_csv, index=False)
    print(f"Wrote summary: {summary_csv}")

if 'errors_df' in globals() and not errors_df.empty:
    errors_df.to_csv(errors_csv, index=False)
    print(f"Wrote errors: {errors_csv}")